# Plot Constructor Example

This notebook demonstrates how to build stacked plots from processor results.

## 1) Set up environment

In [ ]:
# --- Notebook bootstrap: works locally + in Colab ---
from pathlib import Path
import os, sys

def _find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() and (parent / "app").exists():
            return parent
    return p

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

# If running in Colab and repo files aren't present, clone automatically
if IN_COLAB and not (Path.cwd() / "app").exists():
    import subprocess
    if not Path("Polar-lights").exists():
        subprocess.check_call([
            "git", "clone", "-b", "main",
            "https://github.com/Yuri-ga1/Polar-lights.git"
        ])
    os.chdir("Polar-lights")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Install deps in Colab (safe to re-run)
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "poetry"])
    subprocess.check_call(["poetry", "config", "virtualenvs.create", "false"])
    subprocess.check_call(["poetry", "install", "--no-interaction", "--no-ansi"])

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2) User parameters

Set everything in one place: date, optional station codes, SIMURG email and requested plots.

In [ ]:
DATE_START = "2025-11-12"
DATE_END = "2025-11-13 00:00:00"
BASE_DIR = "files"

SIMURG_EMAIL = "Storm_Plotter_Jupyter_Notebook@gmail.com"


# Per-plot PIPELINE params (not matplotlib style params)
PLOT_SPECS = [
    {
        "name": "GIM",
        "params": {
            "product_type": "uqrg",
            "time": ['2025-11-12 00:15:00', '2025-11-12 01:00:00', '2025-11-12 02:00:00'],
        },
    },
    {
        "name": "OMNI",
        "params": {
            "groups": [
                {
                    "title": "IMF and Dst",
                    "fields": ["Bz", "Dst"],
                },
            ],
        },
    },
    {
        "name": "ionosonde",
        "params": {
            "code": None  # or like ["MO155", "IF843"]
        },
    },
    {
        "name": "Cosmic ray",
        "params": {
            "stations": None, # or like ["OULU", "APTY"]
            "station_layout": "single" # "separate" or "single"
        },
    },
]


## 3) Data loader/orchestrator (decomposed)

This helper inspects requested plot names, downloads only required sources, processes them with corresponding Processor classes and returns `processor_results`.

In [ ]:
from app.pipeline.plot_constructor_data_loader import (
    ConstructorDataConfig,
    PlotConstructorDataLoader,
)
from app.visualization import PlotConstructor

## 4) Build processor results for constructor

In [ ]:
loader = PlotConstructorDataLoader(
    ConstructorDataConfig(
        date_start=DATE_START,
        date_end=DATE_END,
        base_dir=BASE_DIR,
        simurg_email=SIMURG_EMAIL,
    )
)

processor_results = loader.load_for_requested_plots(PLOT_SPECS)
plotter = PlotConstructor(processor_results)
plotter.available_plots()

## 5) Plot stacked charts (same order as input list)

In [ ]:
plot_specs_with_range = []
for spec in PLOT_SPECS:
    if isinstance(spec, str):
        plot_specs_with_range.append(spec)
        continue

    spec_with_range = dict(spec)
    params = dict(spec_with_range.get("params", {}))
    params.setdefault("date_start", DATE_START)
    params.setdefault("date_end", DATE_END)
    spec_with_range["params"] = params
    plot_specs_with_range.append(spec_with_range)

plotter.plot(plot_specs_with_range);
